# UMI in MuJoCo - Phase 2: Simulated scene

Builds the simulated "workspace" that replaces the real robot, GoPro and table:

- **Franka Panda** arm with its parallel gripper, taken from DeepMind's `mujoco_menagerie` (UMI itself also ran on Franka/UR5)
- **Table, cup and saucer** - the same task as UMI's cup-arrangement demo, with random object positions on every reset
- **Wrist camera** on the gripper at 224x224 (approximating the GoPro; MuJoCo has no fisheye so we use a wide field of view)
- **`UmiSimEnv`** - a small environment class that returns UMI-style observations: wrist image, end-effector position, rotation (axis-angle) and gripper width

Needs the Phase 1 environment (`/content/umi_env`). If your Colab session was reset, re-run Phase 1 steps 2, 4, 5, 6 and 7 first.

## 1. Paths and check the Phase 1 environment

In [1]:
USE_DRIVE = True

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/umi_mujoco'
else:
    ROOT = '/content/umi_mujoco'
os.makedirs(f'{ROOT}/sim', exist_ok=True)

PY = sys.executable   # Colab's own Python, no Phase 1 environment needed
os.environ.update(ROOT=ROOT, PY=PY)
print('python:', PY, sys.version.split()[0])

Mounted at /content/drive
python: /usr/bin/python3 3.13.15


In [2]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y libegl1 libgl1 libgl1-mesa-dri libglfw3 libosmesa6-dev patchelf > /dev/null
"$PY" -m pip -q install mujoco imageio
"$PY" -c "import mujoco; print('mujoco', mujoco.__version__)"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.2/30.2 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 25.2 MB/s eta 0:00:00
mujoco 3.15.0


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Download the Franka Panda model (only that folder, not all of menagerie)

In [3]:
%%bash
set -e
MEN=/content/mujoco_menagerie
if [ ! -d "$MEN/franka_emika_panda" ]; then
  git clone --depth 1 --filter=blob:none --sparse https://github.com/google-deepmind/mujoco_menagerie.git "$MEN"
  cd "$MEN" && git sparse-checkout set franka_emika_panda
fi
ls "$MEN/franka_emika_panda"
cd "$MEN" && git log -1 --format='menagerie at %h (%ad)'

assets
CHANGELOG.md
hand.xml
LICENSE
mjx_hand.xml
mjx_panda_nohand.xml
mjx_panda.xml
mjx_scene.xml
mjx_single_cube.xml
panda_nohand.xml
panda.png
panda.xml
README.md
scene.xml
menagerie at 0059d43 (Wed Oct 7 13:07:25 2026 -0700)


Cloning into '/content/mujoco_menagerie'...


## 3. Find a working OpenGL backend for camera rendering
Tries `egl` (GPU) then `osmesa` (CPU) with a tiny test model, and remembers the result.

In [4]:
%%bash
cat > /content/gl_test.py <<'EOF'
import mujoco
m = mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom size="1"/></worldbody></mujoco>')
d = mujoco.MjData(m)
r = mujoco.Renderer(m, 64, 64)
mujoco.mj_forward(m, d)
r.update_scene(d)
r.render()
print('ok')
EOF
rm -f /content/gl_backend.txt
for GL in egl osmesa; do
  if MUJOCO_GL=$GL PYOPENGL_PLATFORM=$GL "$PY" /content/gl_test.py > /dev/null 2>&1; then
    echo $GL > /content/gl_backend.txt
    break
  fi
done
if [ -f /content/gl_backend.txt ]; then echo "working OpenGL backend: $(cat /content/gl_backend.txt)"; else echo "NO WORKING GL BACKEND - rerun Phase 1 step 2 and step 10"; fi

working OpenGL backend: egl


In [5]:
GL = open('/content/gl_backend.txt').read().strip()
os.environ['MUJOCO_GL'] = GL
os.environ['PYOPENGL_PLATFORM'] = GL
print('using', GL)

using egl


## 4. Write the simulation code
Three small files go into `/content/umi_sim/`:

- `scene.py` - copies the Panda XML, adds the wrist camera and a TCP (tool centre point) site, then writes the full scene with table, cup, saucer and two viewing cameras
- `env.py` - `UmiSimEnv`: `reset(seed)`, `step(joint_targets, gripper_width)`, `get_obs()`, plus ground-truth object poses for the scripted expert in Phase 3
- `check_scene.py` - runs checks and saves images

In [6]:
%%bash
mkdir -p /content/umi_sim
touch /content/umi_sim/__init__.py

In [7]:
%%writefile /content/umi_sim/scene.py
"""Builds the MuJoCo scene: Franka Panda (mujoco_menagerie) + table + cup + saucer + wrist camera."""
import os
import numpy as np
import xml.etree.ElementTree as ET

# Wrist camera, expressed in the Panda 'hand' frame (z points along the fingers,
# fingers open/close along the hand y axis). xyaxes="0 1 0  1 0 0" makes image-right = hand y
# (so the two fingers appear left/right) and image-up = hand x (so the fingers sit at the bottom).
WRIST_CAM_POS = "0.06 0 0.03"
WRIST_CAM_XYAXES = "0 1 0 1 0 0"
WRIST_CAM_FOVY = "100"
TCP_OFFSET = "0 0 0.1034"      # standard Panda tool-centre-point offset from the hand frame

CUP_RADIUS, CUP_HALF_H = 0.025, 0.03
SAUCER_RADIUS, SAUCER_HALF_H = 0.065, 0.005


def lookat_xyaxes(pos, target, up=(0.0, 0.0, 1.0)):
    """xyaxes string for a camera at `pos` looking at `target`."""
    pos, target, up = (np.asarray(v, dtype=float) for v in (pos, target, up))
    z = pos - target
    z = z / np.linalg.norm(z)
    x = np.cross(up, z)
    x = x / np.linalg.norm(x)
    y = np.cross(z, x)
    return " ".join("%.5f" % v for v in np.concatenate([x, y]))


def patch_panda(panda_dir):
    """Copy panda.xml -> panda_umi.xml adding a wrist camera and a TCP site, and dropping the keyframe
    (the keyframe has the wrong qpos size once the cup/saucer free joints are added)."""
    src = os.path.join(panda_dir, "panda.xml")
    dst = os.path.join(panda_dir, "panda_umi.xml")
    tree = ET.parse(src)
    root = tree.getroot()
    hand = None
    for body in root.iter("body"):
        if body.get("name") == "hand":
            hand = body
            break
    if hand is None:
        raise RuntimeError("could not find <body name='hand'> in panda.xml - menagerie layout changed")
    ET.SubElement(hand, "camera", name="wrist", pos=WRIST_CAM_POS, xyaxes=WRIST_CAM_XYAXES, fovy=WRIST_CAM_FOVY)
    ET.SubElement(hand, "site", name="tcp", pos=TCP_OFFSET, size="0.004", rgba="1 0 0 1", group="4")
    for kf in root.findall("keyframe"):
        root.remove(kf)
    tree.write(dst)
    return dst


def build_scene(menagerie_dir, out_name="umi_scene.xml"):
    """Write umi_scene.xml next to panda_umi.xml (so meshes resolve) and return its path."""
    panda_dir = os.path.join(menagerie_dir, "franka_emika_panda")
    patch_panda(panda_dir)
    front = lookat_xyaxes((1.25, -0.85, 0.75), (0.45, 0.0, 0.05))
    overhead = lookat_xyaxes((0.45, 0.0, 1.3), (0.45, 0.0, 0.0), up=(1.0, 0.0, 0.0))
    xml = f"""<mujoco model="umi_sim_scene">
  <include file="panda_umi.xml"/>
  <statistic center="0.45 0 0.2" extent="1.2"/>
  <visual>
    <headlight ambient="0.45 0.45 0.45" diffuse="0.6 0.6 0.6" specular="0 0 0"/>
    <global offwidth="960" offheight="720"/>
    <quality shadowsize="2048"/>
  </visual>
  <asset>
    <material name="table_mat" rgba="0.76 0.60 0.42 1"/>
    <material name="cup_mat" rgba="0.95 0.95 0.90 1"/>
    <material name="saucer_mat" rgba="0.20 0.45 0.80 1"/>
  </asset>
  <worldbody>
    <light pos="0.5 0 2" dir="0 0 -1" diffuse="0.7 0.7 0.7"/>
    <geom name="floor" type="plane" pos="0 0 -0.75" size="3 3 0.1" rgba="0.55 0.55 0.58 1"/>
    <geom name="table" type="box" pos="0.4 0 -0.02" size="0.55 0.7 0.02" material="table_mat" friction="1 0.005 0.0001"/>
    <body name="cup" pos="0.5 -0.15 {CUP_HALF_H}">
      <freejoint name="cup_joint"/>
      <geom name="cup_geom" type="cylinder" size="{CUP_RADIUS} {CUP_HALF_H}" mass="0.1" material="cup_mat" friction="1.5 0.01 0.001"/>
    </body>
    <body name="saucer" pos="0.5 0.15 {SAUCER_HALF_H}">
      <freejoint name="saucer_joint"/>
      <geom name="saucer_geom" type="cylinder" size="{SAUCER_RADIUS} {SAUCER_HALF_H}" mass="0.08" material="saucer_mat" friction="1 0.01 0.001"/>
    </body>
    <camera name="front" pos="1.25 -0.85 0.75" xyaxes="{front}"/>
    <camera name="overhead" pos="0.45 0 1.3" xyaxes="{overhead}"/>
  </worldbody>
</mujoco>
"""
    out = os.path.join(panda_dir, out_name)
    with open(out, "w") as f:
        f.write(xml)
    return out

Writing /content/umi_sim/scene.py


In [8]:
%%writefile /content/umi_sim/env.py
"""MuJoCo environment that exposes the same kinds of observations UMI's real-robot setup records."""
import numpy as np
import mujoco
from scipy.spatial.transform import Rotation as R

# Standard Franka "ready" pose: gripper pointing straight down above the table.
READY_QPOS = np.array([0.0, -0.785, 0.0, -2.356, 0.0, 1.571, 0.785])
GRIPPER_MAX_WIDTH = 0.08          # metres (2 x 0.04 m finger travel)

CUP_Z = 0.03
SAUCER_Z = 0.005


class UmiSimEnv:
    def __init__(self, xml_path, img_size=224, substeps=10):
        self.model = mujoco.MjModel.from_xml_path(xml_path)
        self.data = mujoco.MjData(self.model)
        self.img_size = img_size
        self.substeps = substeps            # 10 x 2 ms = 50 Hz control
        self.rng = np.random.default_rng(0)
        self._renderers = {}

        m = self.model
        hinge = int(mujoco.mjtJoint.mjJNT_HINGE)
        slide = int(mujoco.mjtJoint.mjJNT_SLIDE)
        hinges = [j for j in range(m.njnt) if int(m.jnt_type[j]) == hinge]
        slides = [j for j in range(m.njnt) if int(m.jnt_type[j]) == slide]
        if len(hinges) != 7 or len(slides) != 2 or m.nu != 8:
            raise RuntimeError(
                "unexpected robot layout: %d hinge joints, %d slide joints, %d actuators (expected 7, 2, 8)"
                % (len(hinges), len(slides), m.nu))
        self.arm_qadr = np.array([m.jnt_qposadr[j] for j in hinges])
        self.finger_qadr = np.array([m.jnt_qposadr[j] for j in slides])

        def _id(kind, name):
            i = mujoco.mj_name2id(m, kind, name)
            if i < 0:
                raise RuntimeError("scene has no %s named '%s'" % (kind, name))
            return i

        self.tcp_id = _id(mujoco.mjtObj.mjOBJ_SITE, "tcp")
        self.cup_id = _id(mujoco.mjtObj.mjOBJ_BODY, "cup")
        self.saucer_id = _id(mujoco.mjtObj.mjOBJ_BODY, "saucer")
        self.cup_qadr = m.jnt_qposadr[_id(mujoco.mjtObj.mjOBJ_JOINT, "cup_joint")]
        self.saucer_qadr = m.jnt_qposadr[_id(mujoco.mjtObj.mjOBJ_JOINT, "saucer_joint")]

    # ------------------------------------------------------------ reset
    def _sample_layout(self):
        cup_xy = np.array([self.rng.uniform(0.40, 0.60), self.rng.uniform(-0.25, -0.08)])
        saucer_xy = np.array([self.rng.uniform(0.40, 0.60), self.rng.uniform(0.08, 0.25)])
        return cup_xy, saucer_xy

    def _set_free_body(self, qadr, xyz, yaw):
        q = self.data.qpos
        q[qadr:qadr + 3] = xyz
        q[qadr + 3:qadr + 7] = [np.cos(yaw / 2), 0.0, 0.0, np.sin(yaw / 2)]   # w x y z

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        m, d = self.model, self.data
        mujoco.mj_resetData(m, d)
        d.qpos[self.arm_qadr] = READY_QPOS
        d.qpos[self.finger_qadr] = 0.04
        d.ctrl[:7] = READY_QPOS
        d.ctrl[7] = 255.0                    # gripper fully open
        cup_xy, saucer_xy = self._sample_layout()
        self._set_free_body(self.cup_qadr, (cup_xy[0], cup_xy[1], CUP_Z), self.rng.uniform(-np.pi, np.pi))
        self._set_free_body(self.saucer_qadr, (saucer_xy[0], saucer_xy[1], SAUCER_Z), self.rng.uniform(-np.pi, np.pi))
        mujoco.mj_forward(m, d)
        for _ in range(200):                  # let objects settle
            mujoco.mj_step(m, d)
        return self.get_obs()

    # ------------------------------------------------------------ observations
    def render(self, camera, height=None, width=None):
        h = height or self.img_size
        w = width or self.img_size
        if (h, w) not in self._renderers:
            self._renderers[(h, w)] = mujoco.Renderer(self.model, h, w)
        r = self._renderers[(h, w)]
        r.update_scene(self.data, camera=camera)
        return r.render().copy()

    @property
    def arm_q(self):
        return self.data.qpos[self.arm_qadr].copy()

    @property
    def gripper_width(self):
        return float(self.data.qpos[self.finger_qadr].sum())

    def eef_pose(self):
        """TCP position (3,) and rotation as axis-angle (3,), both in the world frame."""
        pos = self.data.site_xpos[self.tcp_id].copy()
        mat = self.data.site_xmat[self.tcp_id].reshape(3, 3).copy()
        return pos, R.from_matrix(mat).as_rotvec()

    def get_obs(self, render=True):
        pos, rotvec = self.eef_pose()
        obs = {
            "robot0_eef_pos": pos,
            "robot0_eef_rot_axis_angle": rotvec,
            "robot0_gripper_width": np.array([self.gripper_width]),
        }
        if render:
            obs["camera0_rgb"] = self.render("wrist")
        return obs

    def get_object_state(self):
        """Ground-truth object poses (privileged info, only for the scripted expert, never for the policy)."""
        return {
            "cup_pos": self.data.xpos[self.cup_id].copy(),
            "saucer_pos": self.data.xpos[self.saucer_id].copy(),
        }

    # ------------------------------------------------------------ control
    def step(self, arm_q_target, gripper_width, render=True):
        d = self.data
        d.ctrl[:7] = arm_q_target
        d.ctrl[7] = float(np.clip(gripper_width / GRIPPER_MAX_WIDTH, 0.0, 1.0) * 255.0)
        for _ in range(self.substeps):
            mujoco.mj_step(self.model, d)
        return self.get_obs(render=render)

    def close(self):
        for r in self._renderers.values():
            r.close()
        self._renderers = {}

Writing /content/umi_sim/env.py


In [9]:
%%writefile /content/umi_sim/check_scene.py
"""Self-check for the simulated scene. Usage: python check_scene.py <menagerie_dir> <output_prefix>"""
import sys, time, traceback
import numpy as np
import imageio

sys.path.insert(0, "/content")
from umi_sim.scene import build_scene
from umi_sim.env import UmiSimEnv, READY_QPOS, CUP_Z, SAUCER_Z

men_dir, prefix = sys.argv[1], sys.argv[2]

try:
    xml_path = build_scene(men_dir)
    env = UmiSimEnv(xml_path)
    env.reset(seed=0)
except Exception:
    traceback.print_exc()
    print("\nSCENE LOAD FAILED - paste the error above to Claude.")
    sys.exit(1)

m = env.model
print("model loaded: nq=%d nv=%d nu=%d nbody=%d ngeom=%d ncam=%d" % (m.nq, m.nv, m.nu, m.nbody, m.ngeom, m.ncam))
print("bodies :", [m.body(i).name for i in range(m.nbody)])
print("joints :", [m.joint(i).name for i in range(m.njnt)])
print("cameras:", [m.camera(i).name for i in range(m.ncam)])
print("actuators:", [m.actuator(i).name for i in range(m.nu)])
print()

results = []


def run(name, fn):
    try:
        results.append((name, True, fn() or ""))
    except Exception as e:
        results.append((name, False, "%s: %s" % (type(e).__name__, e)))


def t_objects_settle():
    env.reset(seed=1)
    s = env.get_object_state()
    cup_up = env.data.xmat[env.cup_id].reshape(3, 3)[2, 2]
    assert abs(s["cup_pos"][2] - CUP_Z) < 0.005, "cup z=%.3f (expected %.3f)" % (s["cup_pos"][2], CUP_Z)
    assert abs(s["saucer_pos"][2] - SAUCER_Z) < 0.005, "saucer z=%.3f" % s["saucer_pos"][2]
    assert cup_up > 0.98, "cup tipped over (up-axis %.2f)" % cup_up
    return "cup z=%.3f saucer z=%.3f cup upright=%.3f" % (s["cup_pos"][2], s["saucer_pos"][2], cup_up)


def t_tcp_pose():
    env.reset(seed=1)
    pos, _ = env.eef_pose()
    zaxis = env.data.site_xmat[env.tcp_id].reshape(3, 3)[:, 2]
    assert zaxis[2] < -0.95, "gripper not pointing down (tcp z-axis = %s)" % np.round(zaxis, 2)
    assert 0.2 < pos[0] < 0.5 and abs(pos[1]) < 0.1, "unexpected TCP position %s" % np.round(pos, 3)
    return "TCP at %s, pointing down" % np.round(pos, 3)


def t_gripper():
    env.reset(seed=1)
    out = []
    for target in (0.0, 0.08, 0.03):
        for _ in range(60):
            env.step(READY_QPOS, target, render=False)
        out.append((target, env.gripper_width))
    assert out[0][1] < 0.01, "closing failed: width %.3f" % out[0][1]
    assert out[1][1] > 0.07, "opening failed: width %.3f" % out[1][1]
    assert abs(out[2][1] - 0.03) < 0.01, "width control inaccurate: asked 0.03, got %.3f" % out[2][1]
    return "commanded->measured " + ", ".join("%.2f->%.3f" % o for o in out)


def t_arm_tracking():
    env.reset(seed=1)
    target = READY_QPOS + np.array([0.3, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
    for _ in range(150):
        env.step(target, 0.08, render=False)
    err = np.abs(env.arm_q - target).max()
    assert err < 0.05, "joint tracking error %.3f rad" % err
    return "max joint error %.4f rad after 3 s" % err


def t_speed():
    env.reset(seed=1)
    t0 = time.time()
    for _ in range(20):
        env.step(READY_QPOS, 0.08, render=True)
    dt = (time.time() - t0) / 20
    return "%.1f ms per step incl. wrist render (%.0f steps/s)" % (dt * 1000, 1 / dt)


run("objects rest on table", t_objects_settle)
run("TCP pose at ready position", t_tcp_pose)
run("gripper width control", t_gripper)
run("arm joint tracking", t_arm_tracking)
run("step speed", t_speed)

# ---- images
env.reset(seed=0)
front = env.render("front", 360, 480)
over = env.render("overhead", 360, 480)
imageio.imwrite(prefix + "_overview.png", np.concatenate([front, over], axis=1))
top, bottom = [], []
for s in range(4):
    env.reset(seed=s)
    top.append(env.render("overhead", 224, 224))
    bottom.append(env.render("wrist", 224, 224))
imageio.imwrite(prefix + "_resets.png", np.concatenate([np.concatenate(top, 1), np.concatenate(bottom, 1)], 0))

print("RESULTS")
for name, ok, msg in results:
    print("  [%s] %-28s %s" % ("PASS" if ok else "FAIL", name, msg))
n_fail = sum(1 for r in results if not r[1])
print("\n%d/%d checks passed. images: %s_overview.png, %s_resets.png" % (len(results) - n_fail, len(results), prefix, prefix))
env.close()

Writing /content/umi_sim/check_scene.py


## 5. Run the scene checks
Loads the scene, then checks that objects rest on the table, the gripper points down, the gripper width and arm joints follow commands, and measures simulation speed. Saves two images.

In [10]:
%%bash
cd /content
export PYTHONPATH=/content
export MUJOCO_GL=$(cat /content/gl_backend.txt)
export PYOPENGL_PLATFORM=$(cat /content/gl_backend.txt)
"$PY" /content/umi_sim/check_scene.py /content/mujoco_menagerie /content/scene_check

model loaded: nq=23 nv=21 nu=8 nbody=14 ngeom=85 ncam=3
bodies : ['world', 'link0', 'link1', 'link2', 'link3', 'link4', 'link5', 'link6', 'link7', 'hand', 'left_finger', 'right_finger', 'cup', 'saucer']
joints : ['joint1', 'joint2', 'joint3', 'joint4', 'joint5', 'joint6', 'joint7', 'finger_joint1', 'finger_joint2', 'cup_joint', 'saucer_joint']
cameras: ['front', 'overhead', 'wrist']
actuators: ['actuator1', 'actuator2', 'actuator3', 'actuator4', 'actuator5', 'actuator6', 'actuator7', 'actuator8']

RESULTS
  [PASS] objects rest on table        cup z=0.030 saucer z=0.005 cup upright=1.000
  [PASS] TCP pose at ready position   TCP at [ 0.306 -0.     0.484], pointing down
  [PASS] gripper width control        commanded->measured 0.00->0.000, 0.08->0.080, 0.03->0.030
  [PASS] arm joint tracking           max joint error 0.0063 rad after 3 s
  [PASS] step speed                   359.7 ms per step incl. wrist render (3 steps/s)

5/5 checks passed. images: /content/scene_check_overview.png, /c

## 6. Look at the scene

In [ ]:
from IPython.display import Image, display
print('Left: front camera.  Right: top-down camera.')
display(Image('/content/scene_check_overview.png'))
print('Top row: top-down view of 4 random resets.  Bottom row: what the wrist camera sees in each.')
display(Image('/content/scene_check_resets.png'))

**What to look for in the images:**
- The robot stands on the table, gripper pointing down, with a white cup on one side and a blue saucer on the other.
- The cup and saucer move between resets (top row).
- In the wrist view (bottom row) you should see the two gripper fingers at the bottom edge and the table with the objects ahead of them. If the fingers are missing, cut off or the view looks wrong, tell me what you see and I will adjust the camera position (`WRIST_CAM_POS` in `scene.py`).

## 7. Save the code to Drive

In [ ]:
import shutil
dst = f'{ROOT}/sim/umi_sim'
shutil.copytree('/content/umi_sim', dst, dirs_exist_ok=True)
print('saved to', dst)


## Phase 2 checklist

- [ ] Step 3 found a working backend (`egl` or `osmesa`)
- [ ] Step 5 shows `5/5 checks passed` (or paste the FAIL lines to Claude)
- [ ] Step 6 images look right: robot on table, cup + saucer move between resets, fingers visible in the wrist view
- [ ] Note the `step speed` number - it decides how long demo collection will take in Phase 3

Next: Phase 3 - a scripted expert that picks up the cup and places it on the saucer, recording 100-300 demonstrations.